# 面试题：不用扩散库，怎样实现 DDPM 的加噪、噪声预测与反向去噪？

## 可以直接复述的回答

DDPM 先定义逐步 beta 噪声计划，并用 alpha_bar 的闭式公式在任意时刻直接采样 x_t。训练网络输入 x_t 和 timestep，目标是实际加入的 epsilon，通常优化均方误差。预测到 epsilon 后可反推出 x0，完整采样则从高斯噪声按时间反向迭代。实现最容易出错的是把单步 alpha_t 当成累计 alpha_bar_t，这会让训练分布和反向公式不一致。时间条件必须真正进入 forward，否则同一数值在不同噪声强度下无法正确解释。评估可以比较“直接把 x_t 当信号”的基线与预测 x0 的重建 MSE，并查看不同 timestep 的噪声标准差。本题用 6 条 16 点设备振动波形、8 步 schedule 和手写 MLP denoiser 做真实 backward 与确定性反向轨迹。

## 真实案例

六条序列模拟六台电机在稳定工况下的一维振动周期，长度统一为 16。数据由不同相位和幅度的脱敏解析波形生成，只用于验证扩散数学，不代表真实生成质量。

In [1]:
from pprint import pprint  # 导入结构化打印函数以展示波形和误差
import math  # 导入正弦函数以构造可读周期信号
import torch  # 导入 PyTorch 以实现真实扩散训练
torch.manual_seed(37)  # 固定噪声和参数初始化保证可复现
torch.set_num_threads(1)  # 限制 CPU 线程以稳定小实验
signals = []  # 创建六台设备的干净波形列表
for machine_index in range(6):  # 遍历六台脱敏设备
    phase = machine_index * 0.25  # 给每台设备设置不同相位
    amplitude = 0.75 + machine_index * 0.05  # 给每台设备设置轻微不同幅度
    values = [amplitude * math.sin(2 * math.pi * step / 16 + phase) + 0.15 * math.sin(4 * math.pi * step / 16) for step in range(16)]  # 生成带二次谐波的周期振动
    signals.append(values)  # 保存当前设备波形
x0 = torch.tensor(signals, dtype=torch.float32)  # 构造六乘十六干净数据张量
step_count = 8  # 设置教学扩散总步数
betas = torch.linspace(0.02, 0.16, step_count)  # 定义逐渐增大的线性噪声计划
alphas = 1.0 - betas  # 计算每一步保留信号的比例
alpha_bars = torch.cumprod(alphas, dim=0)  # 计算从零到当前步的累计信号比例
print("六台设备干净振动输入：")  # 输出真实案例标题
pprint([{"设备": f"M{index + 1}", "前8点": [round(value, 3) for value in signal[:8]], "幅值": round(max(signal) - min(signal), 3)} for index, signal in enumerate(signals)])  # 展示六条可读波形
print("噪声 schedule：")  # 输出 schedule 标题
pprint([{"t": index, "beta": round(float(betas[index]), 4), "alpha": round(float(alphas[index]), 4), "alpha_bar": round(float(alpha_bars[index]), 4)} for index in range(step_count)])  # 展示单步与累计系数

六台设备干净振动输入：
[{'前8点': [0.0, 0.393, 0.68, 0.799, 0.75, 0.587, 0.38, 0.181],
  '幅值': 1.598,
  '设备': 'M1'},
 {'前8点': [0.198, 0.586, 0.838, 0.898, 0.775, 0.534, 0.258, 0.008],
  '幅值': 1.673,
  '设备': 'M2'},
 {'前8点': [0.408, 0.768, 0.966, 0.951, 0.746, 0.427, 0.089, -0.197],
  '幅值': 1.712,
  '设备': 'M3'},
 {'前8点': [0.613, 0.925, 1.049, 0.949, 0.659, 0.268, -0.118, -0.421],
  '幅值': 1.799,
  '设备': 'M4'},
 {'前8点': [0.799, 1.041, 1.078, 0.886, 0.513, 0.062, -0.352, -0.648],
  '幅值': 1.907,
  '设备': 'M5'},
 {'前8点': [0.949, 1.103, 1.044, 0.761, 0.315, -0.178, -0.598, -0.862],
  '幅值': 2.052,
  '设备': 'M6'}]
噪声 schedule：
[{'alpha': 0.98, 'alpha_bar': 0.98, 'beta': 0.02, 't': 0},
 {'alpha': 0.96, 'alpha_bar': 0.9408, 'beta': 0.04, 't': 1},
 {'alpha': 0.94, 'alpha_bar': 0.8844, 'beta': 0.06, 't': 2},
 {'alpha': 0.92, 'alpha_bar': 0.8136, 'beta': 0.08, 't': 3},
 {'alpha': 0.9, 'alpha_bar': 0.7322, 'beta': 0.1, 't': 4},
 {'alpha': 0.88, 'alpha_bar': 0.6444, 'beta': 0.12, 't': 5},
 {'alpha': 0.86, 'alpha_bar'

## Baseline / 基线：把带噪 x_t 直接当作重建

在固定 t=7 使用同一组 epsilon 加噪，基线不做任何去噪，直接输出 x_t。逐设备 MSE 将与网络重建在完全相同的噪声样本上比较。

In [2]:
def q_sample(clean, timesteps, noise):  # 定义 DDPM 任意时刻闭式前向加噪
    signal_scale = alpha_bars[timesteps].sqrt().unsqueeze(1)  # 读取每条样本累计信号系数
    noise_scale = (1.0 - alpha_bars[timesteps]).sqrt().unsqueeze(1)  # 读取每条样本累计噪声系数
    return signal_scale * clean + noise_scale * noise  # 按闭式公式合成 x_t
evaluation_noise = torch.randn_like(x0)  # 固定六条评估噪声
evaluation_timesteps = torch.full((len(signals),), step_count - 1, dtype=torch.long)  # 把六条样本都加噪到最后一步
evaluation_xt = q_sample(x0, evaluation_timesteps, evaluation_noise)  # 生成同一评测使用的高噪声输入
baseline_mse_per_signal = ((evaluation_xt - x0) ** 2).mean(dim=1)  # 计算直接输出带噪信号的逐设备 MSE
print("带噪 Baseline 逐设备 MSE：")  # 输出基线标题
pprint([{"设备": f"M{index + 1}", "MSE": round(float(value), 4), "x_t前4点": [round(float(item), 3) for item in evaluation_xt[index, :4]]} for index, value in enumerate(baseline_mse_per_signal)])  # 展示误差和带噪输入
print(f"Baseline mean MSE={float(baseline_mse_per_signal.mean()):.4f}")  # 输出基线平均误差

带噪 Baseline 逐设备 MSE：
[{'MSE': 0.5958, 'x_t前4点': [-0.069, -0.094, 1.168, 0.833], '设备': 'M1'},
 {'MSE': 0.6927, 'x_t前4点': [-0.325, 0.367, -0.329, 0.242], '设备': 'M2'},
 {'MSE': 0.6933, 'x_t前4点': [0.985, 0.69, -0.619, 2.541], '设备': 'M3'},
 {'MSE': 0.2958, 'x_t前4点': [0.808, 1.051, 1.4, 0.767], '设备': 'M4'},
 {'MSE': 0.3362, 'x_t前4点': [1.397, -0.728, 1.398, 1.195], '设备': 'M5'},
 {'MSE': 0.4288, 'x_t前4点': [1.339, 0.84, 0.444, 0.553], '设备': 'M6'}]
Baseline mean MSE=0.5071


## 手写 timestep 条件 MLP denoiser

时间特征包含归一化 t、sin 和 cos，和 x_t 拼接后经过两层 tanh MLP 输出 16 维 epsilon。网络结构、forward 和 x0 反推公式都直接写出，没有调用扩散框架。

In [3]:
def time_features(timesteps):  # 定义三维确定性 timestep 编码
    normalized = timesteps.to(torch.float32) / (step_count - 1)  # 把离散时间缩放到零一
    angle = normalized * math.pi  # 把归一化时间映射到半周期角度
    return torch.stack([normalized, torch.sin(angle), torch.cos(angle)], dim=1)  # 返回 t、sin 和 cos 三维条件
class TinyNoisePredictor(torch.nn.Module):  # 定义最小时间条件噪声预测器
    def __init__(self, signal_size=16, hidden_size=64):  # 初始化两层 MLP 参数
        super().__init__()  # 初始化 PyTorch 模块基类
        self.input_weight = torch.nn.Parameter(torch.randn(signal_size + 3, hidden_size) * 0.12)  # 创建带时间条件的输入权重
        self.input_bias = torch.nn.Parameter(torch.zeros(hidden_size))  # 创建隐藏层偏置
        self.output_weight = torch.nn.Parameter(torch.randn(hidden_size, signal_size) * 0.12)  # 创建噪声输出权重
        self.output_bias = torch.nn.Parameter(torch.zeros(signal_size))  # 创建噪声输出偏置
    def forward(self, noisy_signal, timesteps):  # 定义 epsilon_theta(x_t,t) 前向过程
        conditioned_input = torch.cat([noisy_signal, time_features(timesteps)], dim=1)  # 拼接带噪序列与时间编码
        hidden = torch.tanh(conditioned_input @ self.input_weight + self.input_bias)  # 计算非线性隐藏表示
        return hidden @ self.output_weight + self.output_bias  # 输出与输入同长度的预测噪声
def predict_x0(noisy_signal, timesteps, predicted_noise):  # 定义由 epsilon 反推干净信号的公式
    signal_scale = alpha_bars[timesteps].sqrt().unsqueeze(1)  # 读取累计信号系数
    noise_scale = (1.0 - alpha_bars[timesteps]).sqrt().unsqueeze(1)  # 读取累计噪声系数
    return (noisy_signal - noise_scale * predicted_noise) / signal_scale  # 根据闭式前向公式解出 x0
model = TinyNoisePredictor()  # 实例化手写 denoiser
with torch.no_grad():  # 关闭输入检查阶段梯度记录
    initial_prediction = model(evaluation_xt, evaluation_timesteps)  # 运行未训练噪声预测
print("denoiser 张量：", {"input": tuple(evaluation_xt.shape), "time": tuple(time_features(evaluation_timesteps).shape), "epsilon_hat": tuple(initial_prediction.shape)})  # 展示核心前向张量形状

denoiser 张量： {'input': (6, 16), 'time': (6, 3), 'epsilon_hat': (6, 16)}


## 固定训练噪声库与真实 backward

为保证 Notebook 每次输出一致，预先为每台设备、每个 timestep 生成 24 份固定噪声，共 1152 个训练样本。loss 是预测 epsilon 与真实 epsilon 的均方误差，评估噪声不在训练库中。

In [4]:
noise_repeats = 24  # 为每个设备和 timestep 准备二十四种噪声实现
training_clean = x0.repeat_interleave(step_count * noise_repeats, dim=0)  # 为每条干净波形复制八步乘二十四份
training_timesteps = torch.arange(step_count, dtype=torch.long).repeat_interleave(noise_repeats).repeat(len(signals))  # 构造每条设备都覆盖全部时间和噪声重复的标签
training_noise = torch.randn_like(training_clean)  # 生成固定训练噪声库
training_noisy = q_sample(training_clean, training_timesteps, training_noise)  # 用正确 alpha_bar 闭式生成全部 x_t
optimizer = torch.optim.Adam(model.parameters(), lr=0.012)  # 使用基础 Adam 更新手写 MLP 参数
training_ledger = []  # 创建噪声预测 loss 与梯度账本
for epoch in range(801):  # 执行八百零一次确定性全批训练
    optimizer.zero_grad()  # 清空上一轮梯度
    predicted_noise = model(training_noisy, training_timesteps)  # 运行真实 epsilon 预测 forward
    loss = ((predicted_noise - training_noise) ** 2).mean()  # 手写噪声预测均方误差
    loss.backward()  # 运行真实 backward 计算全部参数梯度
    gradient_norm = float(model.input_weight.grad.norm())  # 读取输入层梯度范数
    if epoch % 160 == 0:  # 每一百六十轮记录一次训练状态
        training_ledger.append({"epoch": epoch, "noise_mse": round(float(loss), 6), "input_grad": round(gradient_norm, 6)})  # 保存 loss 和真实梯度
    optimizer.step()  # 根据当前梯度更新手写 denoiser 参数
print("DDPM 噪声预测训练账本：")  # 输出训练过程标题
pprint(training_ledger)  # 展示噪声 MSE 与梯度随训练变化

DDPM 噪声预测训练账本：
[{'epoch': 0, 'input_grad': 0.28289, 'noise_mse': 1.119802},
 {'epoch': 160, 'input_grad': 0.028215, 'noise_mse': 0.085063},
 {'epoch': 320, 'input_grad': 0.102522, 'noise_mse': 0.062656},
 {'epoch': 480, 'input_grad': 0.049509, 'noise_mse': 0.055678},
 {'epoch': 640, 'input_grad': 0.004431, 'noise_mse': 0.051206},
 {'epoch': 800, 'input_grad': 0.001906, 'noise_mse': 0.048357}]


## 逐设备重建、反向轨迹与结果解读

模型先预测评估噪声，再由累计系数反推 x0。随后从 M1 的 x7 开始，用同一预测 epsilon 构造确定性逐步轨迹，输出每步与干净信号的 MSE；这不是随机 DDPM 采样器，而是便于观察公式的教学反演。

In [5]:
with torch.no_grad():  # 关闭评估阶段梯度记录
    evaluation_epsilon_hat = model(evaluation_xt, evaluation_timesteps)  # 预测六条高噪声输入的 epsilon
    reconstructed_x0 = predict_x0(evaluation_xt, evaluation_timesteps, evaluation_epsilon_hat)  # 按正确累计系数反推干净信号
reconstruction_mse = ((reconstructed_x0 - x0) ** 2).mean(dim=1)  # 计算逐设备重建 MSE
result_rows = []  # 创建同噪声逐设备结果表
for index in range(len(signals)):  # 遍历六台设备
    result_rows.append({"设备": f"M{index + 1}", "Noisy Baseline MSE": round(float(baseline_mse_per_signal[index]), 4), "Denoised MSE": round(float(reconstruction_mse[index]), 4), "重建前4点": [round(float(value), 3) for value in reconstructed_x0[index, :4]]})  # 保存基线、重建误差和波形片段
mean_reconstruction_mse = float(reconstruction_mse.mean())  # 计算模型平均重建误差
trajectory = []  # 创建从最后时刻向前的确定性反演轨迹
current = evaluation_xt[0:1]  # 选择 M1 的 x7 作为轨迹起点
for timestep in range(step_count - 1, -1, -1):  # 从 t=7 逐步走到 t=0
    timestep_tensor = torch.tensor([timestep], dtype=torch.long)  # 构造当前单样本 timestep 张量
    with torch.no_grad():  # 关闭反向轨迹的梯度记录
        epsilon_hat = model(current, timestep_tensor)  # 预测当前状态噪声
        x0_hat = predict_x0(current, timestep_tensor, epsilon_hat)  # 估计当前干净波形
    trajectory.append({"t": timestep, "当前x0估计MSE": round(float(((x0_hat - x0[0:1]) ** 2).mean()), 5)})  # 保存每一步对真实 M1 的误差
    if timestep > 0:  # 还未到零时构造前一时刻状态
        previous_alpha_bar = alpha_bars[timestep - 1]  # 读取前一时刻累计信号比例
        current = previous_alpha_bar.sqrt() * x0_hat + (1 - previous_alpha_bar).sqrt() * epsilon_hat  # 用预测噪声确定性投影到 x_{t-1}
print("逐设备同噪声重建结果：")  # 输出结果表标题
pprint(result_rows)  # 展示模型相对 noisy baseline 的真实数字
print(f"mean MSE 从 {float(baseline_mse_per_signal.mean()):.4f} 到 {mean_reconstruction_mse:.4f}")  # 输出同口径平均误差变化
print("M1 反向时间轨迹：")  # 输出反向过程标题
pprint(trajectory)  # 展示每个 timestep 的干净信号估计误差

逐设备同噪声重建结果：
[{'Denoised MSE': 0.0946,
  'Noisy Baseline MSE': 0.5958,
  '设备': 'M1',
  '重建前4点': [0.226, 0.827, 0.845, 0.568]},
 {'Denoised MSE': 0.0692,
  'Noisy Baseline MSE': 0.6927,
  '设备': 'M2',
  '重建前4点': [0.171, 0.792, 1.07, 1.031]},
 {'Denoised MSE': 0.0416,
  'Noisy Baseline MSE': 0.6933,
  '设备': 'M3',
  '重建前4点': [0.51, 0.649, 1.069, 0.619]},
 {'Denoised MSE': 0.0168,
  'Noisy Baseline MSE': 0.2958,
  '设备': 'M4',
  '重建前4点': [0.593, 0.802, 1.001, 0.698]},
 {'Denoised MSE': 0.0485,
  'Noisy Baseline MSE': 0.3362,
  '设备': 'M5',
  '重建前4点': [0.381, 0.997, 1.045, 0.67]},
 {'Denoised MSE': 0.1609,
  'Noisy Baseline MSE': 0.4288,
  '设备': 'M6',
  '重建前4点': [1.616, 1.484, 1.468, 0.763]}]
mean MSE 从 0.5071 到 0.0719
M1 反向时间轨迹：
[{'t': 7, '当前x0估计MSE': 0.09461},
 {'t': 6, '当前x0估计MSE': 0.07489},
 {'t': 5, '当前x0估计MSE': 0.05193},
 {'t': 4, '当前x0估计MSE': 0.02655},
 {'t': 3, '当前x0估计MSE': 0.01166},
 {'t': 2, '当前x0估计MSE': 0.00929},
 {'t': 1, '当前x0估计MSE': 0.01037},
 {'t': 0, '当前x0估计MSE': 0.01048}]


## 失败案例：误用单步 alpha_t 代替累计 alpha_bar_t

在 t=7，单步 alpha 只描述第七次变换，alpha_bar 才描述前八次噪声累计。错误公式会严重低估噪声标准差，使训练时刻含义和反推公式不一致。

In [6]:
late_timestep = step_count - 1  # 选择累计差异最大的最后时刻
correct_noise_std = float((1 - alpha_bars[late_timestep]).sqrt())  # 计算正确累计噪声标准差
wrong_noise_std = float((1 - alphas[late_timestep]).sqrt())  # 错误地只使用当前单步 alpha
probe_noise = torch.ones_like(x0[0:1])  # 构造全一噪声便于直接比较尺度
correct_probe = alpha_bars[late_timestep].sqrt() * x0[0:1] + correct_noise_std * probe_noise  # 用 alpha_bar 生成正确晚期样本
wrong_probe = alphas[late_timestep].sqrt() * x0[0:1] + wrong_noise_std * probe_noise  # 用单步 alpha 生成错误晚期样本
distribution_gap = float(torch.abs(correct_probe - wrong_probe).mean())  # 计算两个训练分布的平均绝对差
print("失败案例：晚期噪声尺度", {"正确sqrt(1-alpha_bar)": round(correct_noise_std, 4), "错误sqrt(1-alpha_t)": round(wrong_noise_std, 4), "样本平均差": round(distribution_gap, 4)})  # 展示错误公式的实际影响
print("修正原则：前向闭式与 x0 反推始终使用同一份累计 alpha_bar 表")  # 输出修正结论

失败案例：晚期噪声尺度 {'正确sqrt(1-alpha_bar)': 0.7311, '错误sqrt(1-alpha_t)': 0.4, '样本平均差': 0.3311}
修正原则：前向闭式与 x0 反推始终使用同一份累计 alpha_bar 表


## 生产差距

真实扩散模型通常在图像或音频 latent 上使用 U-Net/Transformer、正弦时间嵌入、EMA、混合精度和大规模随机 timestep 训练。采样需实现正确后验均值与方差、DDIM 等加速器，并评估 FID、覆盖率和条件一致性；beta schedule、预测目标与权重必须和部署采样器严格版本一致。

In [7]:
assert len(signals) == 6 and x0.shape == (6, 16)  # 验证真实案例包含六条十六点振动波形
assert training_ledger[-1]["noise_mse"] < training_ledger[0]["noise_mse"]  # 验证真实 backward 降低噪声预测误差
assert mean_reconstruction_mse < float(baseline_mse_per_signal.mean())  # 验证去噪重建优于直接输出 x_t 基线
assert correct_noise_std > wrong_noise_std  # 验证单步 alpha 确实低估晚期累计噪声
assert distribution_gap > 0.1  # 验证错误公式造成可观测训练分布偏移
assert len(trajectory) == step_count  # 验证反向轨迹覆盖全部八个 timestep
print("最小回归测试通过：加噪闭式、时间条件、真实训练、重建与 alpha_bar 修正均满足预期")  # 输出集中断言的验收结论

最小回归测试通过：加噪闭式、时间条件、真实训练、重建与 alpha_bar 修正均满足预期
